# 12 · Every finalist on every scene

The eight finalists of round 2, each with the seed that scored best on validation (finalist 2 is the deployed model, seed 2), and the hand-designed U-Nets up to 6 MB, run on every scene we have: every HEO scene used in this work at 2.5, 5 and 10 m, the PhiSat-2 validation and test scenes at 4.75 m and the three Sentinel-2 test chips. For each scene and resolution there is a page with the image, the ground truth and every model's fresh-burn map (found in yellow, missed in cyan, false alarms in magenta). Under each map are its IoU, precision and recall, and the hectares found, missed and falsely mapped. When a scene has labelled fresh burn, a second page zooms on it.

Counting: on the scene, every pixel once, over labelled pixels. The scores of 08–10 counted window pixels, so the last row and column of windows, which overlap their neighbours, counted twice there; the IoU here can differ in the second decimal. Hectares are pixel counts times the pixel area at that resolution. The input is prepared as on board (`input_mask="image"`).

The test stays closed while the model is chosen. With `UNBLIND = False` (the default) the notebook leaves out every test scene and the test scores: the choice is made on the validation, search and train scenes and on the validation scores, and written down in [9]. Only then does `UNBLIND = True` add the test pages and the test scores of every finalist, for reporting, not for choosing again. Train scenes were learnt from: their maps show fit, not generalisation.

Pages go to `models/pynas/review_finalists/pages/` (one folder per set), the per-scene numbers to `scene_metrics.csv` there.

In [ ]:
# [1] setup
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                # the notebooks run from the repository root
import os, sys, json, gc, math, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch, Rectangle
import torch
from IPython.display import Image, display
for p in (os.path.join(os.getcwd(), "pyqnas", "src"), os.getcwd()):
    if p not in sys.path:
        sys.path.insert(0, p)
import pynas
from pynas import config, final, report, runner
from pynas.hardware import myriad
from pynas.hardware.latency import latency_ms
from pynas.space.network import n_params_used
from pynas.train import data as D
from pynas.train.metrics import SCENE_MIN_PX
import burnscar.dataset as BD
import burnscar.pool as BP

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 300)
cfg = config.load(Path("configs/burnscar.ini"))
cfg2 = config.with_overrides(cfg, {"Final.finalists": "8"})
spec = D.DataSpec.from_config(cfg)
dcfg = json.load(open("data/tables/dataset_config.json"))
scenes = pd.read_csv(dcfg["files"]["scenes"])
norm = json.load(open(dcfg["files"]["normalisation"]))
SIZE = dcfg["size"]
POOL, BASE = Path(spec.pool), Path(cfg.get("Run", "out"))
R2 = BASE / "final_r2"
FQ, Q2F = R2 / "finalists" / "runs" / "queue.json", R2 / "runs" / "queue.json"
REVIEW, OUT = POOL / "review", BASE / "review_finalists"
PAGES = OUT / "pages"
REVIEW.mkdir(exist_ok=True); PAGES.mkdir(parents=True, exist_ok=True)
MAX_MB = 6.0          # hand-designed U-Nets up to this IR size go next to the finalists
PHISAT2 = "all"       # PhiSat-2 validation and test scenes: "all", or the n of each with the most fresh burn (10 used 6)
MAX_NEW_GB = 40       # new windows are written only below this size
UNBLIND = True       # True only after the choice is written down in [9]: test pages and test scores
CORES = len(os.sched_getaffinity(0))
dev = "cuda:0" if torch.cuda.is_available() else "cpu"
for f in (FQ, Q2F, R2 / "finalists.json", Path("models/onboard/burnscar_pynas_r2/deploy.json")):
    assert Path(f).exists(), f"{f} is missing: finish 09 first"
print(f"PyNAS {pynas.__version__}; device {dev}")

In [ ]:
# [2] the models: every finalist with its best seed on validation, and the hand-designed U-Nets up to MAX_MB
pick = json.load(open(R2 / "finalists.json"))
dep = json.load(open("models/onboard/burnscar_pynas_r2/deploy.json"))
ft = pd.DataFrame(pick["table"]).set_index("label")
res = final.results(FQ)
res["val"] = [final._val_score(r, cfg2) for _, r in res.iterrows()]
rows = []
for label, g in sorted(res.groupby("label"), key=lambda t: int(str(t[0]).split()[-1])):
    b, f = g.loc[g.val.idxmax()], ft.loc[label]
    k = int(str(label).split()[-1])
    rows.append(dict(name=f"F{k}" + (" (chosen)" if label == pick["label"] else ""), kind="finalist", label=label,
                     spec=f.spec, ir_mb=float(f.ir_mb), latency_ms=float(f.latency_ms), params_M=float(f.params) / 1e6,
                     val_mean=float(f.val_mean), val_se=float(f.val_se), seed=int(b.seed), run_dir=FQ.parent / b.run))

res2 = final.results(Q2F)
base = res2[res2.role == "baseline"].copy()
base["val"] = [final._val_score(r, cfg2) for _, r in base.iterrows()]
bb = base.loc[base.groupby("label").val.idxmax()].reset_index(drop=True)
nets_b = {f"b{i}": runner.load_result(R2 / "runs" / r.run) for i, r in bb.iterrows()}
shape = tuple(json.load(open(R2 / "runs" / bb.run.iloc[0] / "run.json"))["input_shape"])
hw = myriad.check(nets_b, R2 / "hardware", shape, cfg2) if myriad.status(cfg2)["model_optimizer"] else {}
lat = latency_ms(nets_b, shape, cfg2) if myriad.status(cfg2)["nnmeter"] else {}
for i, r in bb.iterrows():
    k = f"b{i}"
    params = n_params_used(nets_b[k], shape)
    ir = hw.get(k, {}).get("ir_mb") or 2 * params / 1e6
    print(f"{report.short_label(r.label)}: {ir:.2f} MB" + ("" if ir <= MAX_MB else f", over {MAX_MB:g} MB: left out"))
    if ir <= MAX_MB:
        rows.append(dict(name=report.short_label(r.label), kind="hand-designed", label=r.label, spec=r.label,
                         ir_mb=float(ir), latency_ms=float(lat.get(k, np.nan)), params_M=params / 1e6,
                         val_mean=float(base[base.label == r.label].val.mean()), val_se=np.nan, seed=int(r.seed),
                         run_dir=R2 / "runs" / r.run))
del nets_b
M = pd.DataFrame(rows)
models = {r.name: (runner.load_result(r.run_dir, dev), "fp16") for r in M.itertuples()}
TITLE = {r.name: f"{r.name}\n{r.ir_mb:.2f} MB · seed {r.seed}" for r in M.itertuples()}
chosen = M[M.label == pick["label"]].iloc[0]
print(f"chosen finalist shown with seed {chosen.seed}; deployed: seed {dep['seed']}")
M[["name", "kind", "seed", "ir_mb", "params_M", "latency_ms", "val_mean", "val_se", "spec"]].round(3)

## The scenes and their windows

Every HEO scene with windows in the training pool or in a scoring set, with its role, at 2.5, 5 and 10 m (the windows written in 10 are used again); the PhiSat-2 validation and test scenes (`PHISAT2`); the Sentinel-2 test chips as stored.

In [ ]:
# [3] the scenes, and their windows written once
def scenes_of(name):
    return set(pd.read_csv(POOL / f"{name}_windows.csv", usecols=["scene"]).scene)


role = {}
for name, r in [("heo_test", "test"), ("heo_val", "val"), ("heo_search", "search"), ("train", "train")]:
    for sc in scenes_of(name):
        role.setdefault(sc, r)
heo = scenes[(scenes.source == "heo") & scenes.scene.isin(role)].assign(role=lambda t: t.scene.map(role))
ph = []
for name, r in [("phisat2_val", "val"), ("phisat2_test", "test")]:
    w = pd.read_csv(POOL / f"{name}_windows.csv", usecols=["scene", "fresh"])
    order = w.groupby("scene").fresh.sum().sort_values(ascending=False).index
    ph.append(scenes[scenes.scene.isin(order if PHISAT2 == "all" else order[:int(PHISAT2)])].assign(role=r))
ph = pd.concat(ph)
PH = "phisat2_g4.75" if PHISAT2 == 6 else f"phisat2_{PHISAT2}_g4.75"
SETS = {"heo_g2.5": (heo, [2.5]), "heo_g5": (heo, [5.0]), "heo_g10": (heo, [10.0]), PH: (ph, [4.75])}
display(pd.concat([heo, ph]).groupby(["source", "role"]).scene.nunique().rename("scenes").to_frame())
for name, (tab, gsds) in SETS.items():
    prefix = REVIEW / name
    if Path(f"{prefix}_windows.csv").exists():
        continue
    w = BP.grid_windows(tab, gsds, SIZE).reset_index(drop=True)
    w["role"] = w.scene.map(dict(zip(tab.scene, tab.role)))
    gb = len(w) * SIZE * SIZE * 17 / 1e9
    print(f"{name}: {len(w):,} windows from {w.scene.nunique()} scenes, about {gb:.1f} GB")
    assert gb < MAX_NEW_GB, f"{name} would take about {gb:.0f} GB: raise MAX_NEW_GB or set PHISAT2 to a number"
    BP.write_set(BD.FixedWindows(w, scenes, norm, size=SIZE), str(prefix), workers=max(1, CORES - 1), every=0,
                 extra={"fresh": w.fresh.values, "old": w.old.values, "role": w.role.values})
LOOP = [(f"review/{n}", "HEO" if n.startswith("heo") else "PhiSat-2 (sim.)") for n in SETS]
LOOP += [(n, "Sentinel-2") for n in spec.test if n.startswith("s2")] if UNBLIND else []
LOOP

## Numbers and pages

One page per scene and resolution: the image (NIR, red, green), the ground truth, and every model's fresh-burn map. Under each map: IoU, precision (P) and recall (R); the hectares found (hits), missed and falsely mapped. The zoom page crops all panels to the labelled fresh burn, at full resolution.

In [ ]:
# [4] numbers and pages of one scene at one resolution
NAMES = list(spec.class_names)
FRESH = spec.target
OLD = NAMES.index("old") if "old" in NAMES else None
CLOUD = NAMES.index("cloud") if "cloud" in NAMES else None
CMAP = ListedColormap([report.CLASS_COLOURS.get(n, "#999999") for n in NAMES])
NCOL, PANEL, MAXPX, ZOOM_MIN, ZOOM_MAX = 6, 2.9, 1000, 192, 900


def ha(v):
    return "–" if not np.isfinite(v) else (f"{v:,.0f}" if v >= 100 else f"{v:.1f}" if v >= 10 else f"{v:.2f}")


def f2(v):
    return "–" if not np.isfinite(v) else f"{v:.2f}"


def stretch(img):
    lo, hi = np.nanpercentile(img, 2, (0, 1)), np.nanpercentile(img, 98, (0, 1))
    return np.clip((img - lo) / np.maximum(hi - lo, 1e-6), 0, 1)


def paste_box(canvas, tiles, r, c, r0, c0):
    """Tiles placed at (r, c) of the scene, cut to the canvas that starts at (r0, c0)."""
    H, W = canvas.shape[:2]
    for t, i, j in zip(tiles, r - r0, c - c0):
        i0, j0, i1, j1 = max(i, 0), max(j, 0), min(i + t.shape[0], H), min(j + t.shape[1], W)
        if i1 > i0 and j1 > j0:
            canvas[i0:i1, j0:j1] = t[i0 - i:i1 - i, j0 - j:j1 - j]
    return canvas


def overlay(p, lab, grey):
    show = np.repeat(grey, 3, -1).copy()
    show[(p == FRESH) & (lab == FRESH)] = report.HIT
    show[(p == FRESH) & (lab != FRESH) & (lab >= 0)] = report.FALSE_ALARM
    show[(p != FRESH) & (lab == FRESH)] = report.MISS
    return show


def page(n_panels, top=0.45, gap=1.0, bottom=0.9):
    """Panels in rows of NCOL, with room (inches) for two-line titles above and three-line notes below each."""
    ncol = min(NCOL, n_panels)
    nrow = -(-n_panels // ncol)
    height = (PANEL + gap) * nrow + top + bottom - gap + 0.1
    fig, axes = plt.subplots(nrow, ncol, figsize=(PANEL * ncol, height), squeeze=False)
    a = (height - top - bottom - (nrow - 1) * gap) / nrow
    fig.subplots_adjust(left=0.005, right=0.995, top=1 - top / height, bottom=bottom / height, wspace=0.06,
                        hspace=gap / a if nrow > 1 else 0)
    for ax in axes.flat:
        ax.set_xticks([]), ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)
    for ax in axes.flat[n_panels:]:
        ax.axis("off")
    return fig, axes.flat


def legend(fig):
    h = [Patch(color=report.CLASS_COLOURS.get(n, "#999999"), label=n) for n in NAMES]
    h += [Patch(color=report.HIT, label=f"{spec.target_name}: found"), Patch(color=report.MISS, label="missed"),
          Patch(color=report.FALSE_ALARM, label="false alarm")]
    fig.legend(handles=h, loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=len(h), fontsize=8, frameon=False)


def review_scene(s, preds, sc, g, role_, sensor, rgb=(6, 2, 1)):
    """s: the windows of one scene at one resolution (refl, mask, windows). Returns (one row of numbers per model,
    [the scene page, the zoom page when the scene has labelled fresh burn])."""
    win = s["windows"]
    idx = np.arange(len(win))
    y = report.labels_of(s, spec)
    px_ha = g * g / 1e4
    r1, c1, h1, w1, _ = report._mosaic(idx, win, SIZE, 1, None)
    lab1 = report._paste(np.full((h1, w1), -1, np.int16), y, r1, c1, 1)
    valid, gt = lab1 >= 0, lab1 == FRESH
    n_gt = int(gt.sum())
    info = dict(scene=sc, gsd=g, role=role_, sensor=sensor, labelled_px=n_gt, labelled_ha=n_gt * px_ha,
                area_ha=int(valid.sum()) * px_ha,
                old_ha=int((lab1 == OLD).sum()) * px_ha if OLD is not None else np.nan,
                cloud=int((lab1 == CLOUD).sum()) / max(int(valid.sum()), 1) if CLOUD is not None else np.nan)
    rows = []
    for name, p in preds.items():
        pf = (report._paste(np.full((h1, w1), 255, np.uint8), p, r1, c1, 1) == FRESH) & valid
        tp, fp, fn = int((pf & gt).sum()), int((pf & ~gt).sum()), int((gt & ~pf).sum())
        rows.append(dict(info, model=name, tp_px=tp, fp_px=fp, fn_px=fn, found_ha=tp * px_ha, missed_ha=fn * px_ha,
                         false_alarm_ha=fp * px_ha, mapped_ha=(tp + fp) * px_ha,
                         iou=tp / (tp + fp + fn) if tp + fp + fn else np.nan,
                         precision=tp / (tp + fp) if tp + fp else np.nan, recall=tp / (tp + fn) if tp + fn else np.nan))
    figs = []

    # the scene
    r, c, h, w, step = report._mosaic(idx, win, SIZE, None, MAXPX)
    refl = np.asarray(s["refl"][:, list(rgb)], np.float32) / spec.scale
    img = stretch(report._paste(np.full((h, w, 3), np.nan, np.float32), refl.transpose(0, 2, 3, 1), r, c, step))
    lab = report._paste(np.full((h, w), -1, np.int16), y, r, c, step)
    gaps = (lab < 0) | np.isnan(img).any(-1)
    grey = np.where(gaps[..., None], 1.0, 0.35 + 0.5 * np.nan_to_num(img).mean(-1, keepdims=True))
    fig, ax = page(2 + len(preds))
    ax[0].imshow(np.where(gaps[..., None], 1.0, np.nan_to_num(img)))
    ax[0].set_title(f"{str(sc)[:30]}\n{role_} · {sensor} · {g:g} m", fontsize=8, loc="left")
    ax[0].set_xlabel(f"labelled: {info['area_ha'] / 100:,.1f} km²", fontsize=8)
    ax[1].imshow(np.ma.masked_less(lab, 0), cmap=CMAP, vmin=-0.5, vmax=len(NAMES) - 0.5, interpolation="nearest")
    ax[1].set_title("ground truth", fontsize=8)
    ax[1].set_xlabel(f"fresh burn {ha(info['labelled_ha'])} ha · old {ha(info['old_ha'])} ha\n"
                     f"cloud {info['cloud']:.0%} of the labelled area", fontsize=8)
    for k, (name, p) in enumerate(preds.items()):
        m = rows[k]
        pm = report._paste(np.full((h, w), -1, np.int16), p.astype(np.int16), r, c, step)
        ax[2 + k].imshow(overlay(pm, lab, grey), interpolation="nearest")
        ax[2 + k].set_title(TITLE.get(name, name), fontsize=8)
        if n_gt:
            txt = (f"IoU {f2(m['iou'])} · P {f2(m['precision'])} · R {f2(m['recall'])}\n"
                   f"found {ha(m['found_ha'])} · missed {ha(m['missed_ha'])} ha\n"
                   f"false alarm {ha(m['false_alarm_ha'])} ha")
        else:
            txt = f"no labelled {spec.target_name} burn\nfalse alarm {ha(m['false_alarm_ha'])} ha"
        ax[2 + k].set_xlabel(txt, fontsize=8)
    legend(fig)
    figs.append(fig)

    # the zoom on the labelled fresh burn, at full resolution
    if n_gt >= SCENE_MIN_PX:
        rr, cc = np.nonzero(gt)
        side = max(rr.max() - rr.min(), cc.max() - cc.min(), ZOOM_MIN) * 1.3
        cy, cx = (rr.max() + rr.min()) / 2, (cc.max() + cc.min()) / 2
        R0, C0 = int(max(0, cy - side / 2)), int(max(0, cx - side / 2))
        R1, C1 = int(min(h1, cy + side / 2)), int(min(w1, cx + side / 2))
        z = max(1, math.ceil(max(R1 - R0, C1 - C0) / ZOOM_MAX))
        near = np.nonzero((r1 < R1) & (r1 + SIZE > R0) & (c1 < C1) & (c1 + SIZE > C0))[0]
        zr, zc = r1[near], c1[near]
        zimg = paste_box(np.full((R1 - R0, C1 - C0, 3), np.nan, np.float32), refl[near].transpose(0, 2, 3, 1),
                         zr, zc, R0, C0)[::z, ::z]
        zimg = stretch(zimg)
        zlab = paste_box(np.full((R1 - R0, C1 - C0), -1, np.int16), y[near], zr, zc, R0, C0)[::z, ::z]
        zgaps = (zlab < 0) | np.isnan(zimg).any(-1)
        zgrey = np.where(zgaps[..., None], 1.0, 0.35 + 0.5 * np.nan_to_num(zimg).mean(-1, keepdims=True))
        fig, ax = page(2 + len(preds))
        ax[0].imshow(np.where(zgaps[..., None], 1.0, np.nan_to_num(zimg)))
        ax[0].set_title(f"{str(sc)[:30]} · zoom\n{role_} · {sensor} · {g:g} m", fontsize=8, loc="left")
        ax[0].set_xlabel(f"{(C1 - C0) * g / 1000:.1f} × {(R1 - R0) * g / 1000:.1f} km", fontsize=8)
        ax[1].imshow(np.ma.masked_less(zlab, 0), cmap=CMAP, vmin=-0.5, vmax=len(NAMES) - 0.5, interpolation="nearest")
        ax[1].set_title("ground truth", fontsize=8)
        for k, (name, p) in enumerate(preds.items()):
            zp = paste_box(np.full((R1 - R0, C1 - C0), -1, np.int16), p[near].astype(np.int16), zr, zc, R0, C0)[::z, ::z]
            ax[2 + k].imshow(overlay(zp, zlab, zgrey), interpolation="nearest")
            ax[2 + k].set_title(TITLE.get(name, name), fontsize=8)
            ax[2 + k].set_xlabel(f"IoU on the scene {f2(rows[k]['iou'])}", fontsize=8)
        legend(fig)
        figs.append(fig)
    return rows, figs

In [ ]:
# [5] scene by scene: predictions (input masked by the image only, as on board), numbers and pages
# (scenes already in scene_metrics.csv with the same models are not done again)
ORDER = {"test": 0, "val": 1, "search": 2, "train": 3}
old = pd.read_csv(OUT / "scene_metrics.csv") if (OUT / "scene_metrics.csv").exists() else pd.DataFrame()
if len(old) and set(old.model) != set(models):
    old = pd.DataFrame()
done = set(zip(old.set, old.scene.astype(str), old.gsd)) if len(old) else set()
tables = []
for set_name, sensor in LOOP:
    s = D.open_set(spec, set_name, teacher=False)
    win = s["windows"].reset_index(drop=True)
    win["scene"] = win.scene.astype(str)
    if "role" not in win:
        win["role"] = "test"
    keys = win.groupby(["scene", "gsd"], sort=False).role.first().reset_index()
    keys = keys.assign(o=keys.role.map(ORDER).fillna(9)).sort_values(["o", "scene", "gsd"]).reset_index(drop=True)
    if not UNBLIND:
        keys = keys[keys.role != "test"].reset_index(drop=True)
    folder = PAGES / Path(set_name).name
    folder.mkdir(parents=True, exist_ok=True)
    todo = [(n, k) for n, k in keys.iterrows() if (Path(set_name).name, k.scene, k.gsd) not in done]
    for n, k in todo:
        idx = win.index[(win.scene == k.scene) & (win.gsd == k.gsd)].to_numpy()
        sub = dict(refl=np.asarray(s["refl"][idx]), mask=np.asarray(s["mask"][idx]),
                   windows=win.loc[idx].reset_index(drop=True))
        preds = report.predict_set(models, spec, sub, dev, input_mask="image")
        rows, figs = review_scene(sub, preds, k.scene, k.gsd, k.role, sensor)
        tables.append(pd.DataFrame(rows).assign(set=Path(set_name).name))
        for part, fig in zip(("scene", "zoom"), figs):
            fig.savefig(folder / f"{n:03d}_{k.role}_{k.scene[:31]}_{k.gsd:g}m_{part}.jpg", dpi=130,
                        pil_kwargs={"quality": 85})
            plt.close(fig)
        del sub, preds, figs; gc.collect()
    print(f"{set_name}: {len(todo)} of {len(keys)} scenes × resolutions done now, pages in {folder}")
    del s, win
new = (pd.concat(tables, ignore_index=True).merge(M[["name", "kind", "ir_mb", "seed"]], left_on="model", right_on="name")
       .drop(columns="name") if tables else pd.DataFrame())
T = pd.concat([old, new], ignore_index=True)
if not UNBLIND and len(T):
    T = T[T.role != "test"]
T.to_csv(OUT / "scene_metrics.csv", index=False)
print(f"{len(T):,} rows in {OUT / 'scene_metrics.csv'}")

## Summary

Mean fresh-burn IoU over the scenes of each set and role (scenes with at least 500 labelled fresh-burn pixels), the hectares found, missed and falsely mapped summed over the scenes, then every validation scene (every test scene once the test is opened).

In [ ]:
# [6] per set and role: mean IoU, and hectares summed over the scenes
cols = list(models)
S = T[T.labelled_px >= SCENE_MIN_PX]
iou = S.pivot_table(index=["set", "role"], columns="model", values="iou", aggfunc="mean").reindex(columns=cols)
iou.insert(0, "scenes", S.groupby(["set", "role"]).scene.nunique())
display(iou.round(3))
area = T.groupby(["set", "role", "model"], sort=False)[["found_ha", "missed_ha", "false_alarm_ha", "mapped_ha",
                                                       "labelled_ha"]].sum()
display(area.round(1))
display(S[S.role == ("test" if UNBLIND else "val")].pivot_table(index=["set", "scene", "gsd"], columns="model",
                                                                  values="iou").reindex(columns=cols).round(2))

## Pages

The validation pages here (the test pages once the test is opened); every page is in `models/pynas/review_finalists/pages/`, and `show()` puts those of another role here.

In [ ]:
# [7] the validation pages (the test pages once the test is opened)
for f in sorted(PAGES.glob(f"*/*_{'test' if UNBLIND else 'val'}_*.jpg")):
    display(Image(filename=str(f)))

In [ ]:
# [8] the pages of another role: show("search"), show("train"), or show("val", "heo_g2.5")
def show(role_, set_name=None):
    assert role_ != "test" or UNBLIND, "the test pages stay closed until UNBLIND = True in [1]"
    files = sorted(PAGES.glob(f"{set_name or '*'}/*_{role_}_*.jpg"))
    print(f"{len(files)} pages")
    for f in files:
        display(Image(filename=str(f)))


show("search")

## The choice

The model to deliver, by its name in the table of [2], and why: from the validation scores of [2] and the validation, search and train scenes of [6]–[8]. The cell writes it once, with the time, to `choice.json`; the test pages and the test scores open only after that: set `UNBLIND = True` in [1], then run [1], [3], [5]–[7] and [10] (scenes already done are not done again).

In [ ]:
# [9] the choice, written before the test is opene
CHOICE = dict(model="F4", reason="highest HEO 2.5 m IoU outside the test set (validation scene 0.623, "
                   "search scenes 0.665) and best validation score of the finalists (0.678); under 6 MB; "
                   "latency not binding at 40 s per chip")
f = OUT / "choice.json"
if f.exists():
    print("already written:", json.load(open(f)))
elif CHOICE["model"]:
    assert CHOICE["model"] in models, f"not a model of [2]: {CHOICE['model']}"
    assert not UNBLIND, "the choice is written while the test is still closed"
    rec = dict(CHOICE, time=time.strftime("%Y-%m-%d %H:%M:%S"),
               **M.set_index("name").loc[CHOICE["model"], ["label", "spec", "seed", "ir_mb", "val_mean"]].astype(str).to_dict())
    json.dump(rec, open(f, "w"), indent=1)
    print("written:", rec)
else:
    print("no choice yet")

## Every finalist on the test sets

Every finalist and every seed scored once on the test sets, as the deployed model was in 09 (the windows of the test sets, FP16), written to `final_r2/finalists/` and read back afterwards; the hand-designed U-Nets from 09. Opened only after the choice of [9]; reported, not used to choose again.

In [ ]:
# [10] test scores of every finalist and seed, against IR size (after the choice, with UNBLIND)
assert (OUT / "choice.json").exists(), "write the choice in [9] first"
assert UNBLIND, "set UNBLIND = True in [1] to open the test"
choice = json.load(open(OUT / "choice.json"))["model"]
scores, _ = final.test(FQ, cfg2, out=R2 / "finalists")
t = spec.target_name
TEST = {n: f"{n}_labels_{t}_iou" for n in spec.test}
NICE = {n: ("HEO" if n.startswith("heo") else "PhiSat-2 (sim.)" if n.startswith("phisat2") else "Sentinel-2" if n.startswith("s2") else n)
        for n in spec.test}
fin = scores[scores.precision == "fp16"].merge(M[M.kind == "finalist"][["label", "name", "ir_mb"]], on="label")
ref = pd.read_csv(R2 / "test_scores.csv")
ref = ref[(ref.precision == "fp16")].merge(M[M.kind == "hand-designed"][["label", "name", "ir_mb"]], on="label")
both = pd.concat([fin, ref], ignore_index=True)
agg = both.groupby(["name", "ir_mb"], sort=False)[list(TEST.values())].agg(["mean", "std"])
tab = pd.DataFrame({NICE[n]: [f"{m:.3f} ± {s:.3f}" for m, s in zip(agg[(c, "mean")], agg[(c, "std")])]
                    for n, c in TEST.items()}, index=agg.index)
tab.insert(0, "seeds", both.groupby(["name", "ir_mb"], sort=False).size())
display(tab)

fig, axes = plt.subplots(1, len(TEST), figsize=(5.2 * len(TEST), 4.2), squeeze=False)
for ax, (n, c) in zip(axes[0], TEST.items()):
    for (name, mb), v in zip(agg.index, agg[(c, "mean")]):
        sd = agg.loc[(name, mb), (c, "std")]
        hand = name in set(M[M.kind == "hand-designed"].name)
        colour = "#d9531e" if "chosen" in name else ("0.55" if hand else "#1c2230")
        ax.errorbar(mb, v, yerr=sd, fmt="s" if hand else "o", color=colour, capsize=3, ms=6)
        if name == choice:
            ax.scatter([mb], [v], s=220, facecolors="none", edgecolors="#d9531e", linewidths=1.5, zorder=3)
        short = name.replace(" (chosen)", "*").replace("U-Net ", "").replace(", slim", " slim")
        ax.annotate(short, (mb, v), xytext=(5, 4), textcoords="offset points", fontsize=8)
    ax.set_title(NICE[n], loc="left", fontsize=10)
    ax.set_xlabel("IR size, MB (FP16)")
    ax.set_ylabel("fresh-burn IoU, test (mean ± sd over seeds)" if ax is axes[0, 0] else "")
fig.suptitle("finalists (circles; * the round-2 pick, ringed: the choice of [9]) and hand-designed U-Nets (squares)",
             x=0.01, ha="left", fontsize=10)
plt.tight_layout(); plt.savefig(OUT / "test_vs_size.png", dpi=200); plt.show()